Semantic Search with ChromaDB

SetUp and Load Negative Reviews

In [1]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [2]:
import chromadb
print("ChromaDB installed successfully!")

ChromaDB installed successfully!


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import libraries
import os
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

# Define project paths
PROJECT_PATH = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine"
DATA_PATH = os.path.join(PROJECT_PATH, "data")

# Load negative reviews
negative_path = os.path.join(DATA_PATH, "negative_reviews.csv")
negative_df = pd.read_csv(negative_path)

print("Negative reviews loaded:", negative_df.shape)
negative_df.head()

Mounted at /content/drive
Negative reviews loaded: (11453, 12)


,Unnamed: 0,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,review_length,sentiment
0,0,A2HD75EMZR8QLN,0700099867,123,"[8, 12]",Installing the game was a struggle (because of...,0.0,Pay to unlock content? I don't think so.,1341792000,"07 9, 2012",779,negative
1,2,A1INA0F5CWW3J4,0700099867,"Amazon Shopper ""Mr.Repsol""","[0, 0]",1st shipment received a book instead of the ga...,0.0,Wrong key,1403913600,"06 28, 2014",282,negative
2,7,AQTC623NCESZW,0700099867,Chesty Puller,"[1, 4]",I can't tell you what a piece of dog**** this ...,0.0,Crash 3 is correct name AKA Microsoft,1353715200,"11 24, 2012",728,negative
3,9,A2JLT2WY0F2HVI,0700099867,D. Sweetapple,"[1, 1]",I still haven't figured this one out. Did ever...,1.0,Couldn't get this one to work,1391817600,"02 8, 2014",404,negative
4,13,A248LSBZT4P38V,0700099867,Joseph R. Kennedy,"[0, 0]",I bought this and the key didn't work. It was...,0.0,"It might have been a good game, but I never fo...",1404086400,"06 30, 2014",185,negative


Prepare Documents

In [4]:
# Use first 5000 reviews for faster experimentation
sample_size = 5000

documents = (
    negative_df["reviewText"]
    .dropna()
    .astype(str)
    .head(sample_size)
    .tolist()
)

print("Documents prepared:", len(documents))
print("\nSample document:\n")
print(documents[0][:500])

Documents prepared: 5000

Sample document:

Installing the game was a struggle (because of games for windows live bugs).Some championship races and cars can only be "unlocked" by buying them as an addon to the game. I paid nearly 30 dollars when the game was new. I don't like the idea that I have to keep paying to keep playing.I noticed no improvement in the physics or graphics compared to Dirt 2.I tossed it in the garbage and vowed never to buy another codemasters game. I'm really tired of arcade style rally/racing games anyway.I'll cont


Load Embedding Model

In [5]:
from sentence_transformers import SentenceTransformer

# Load pre-trained embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


🌐 Model Information

Hum ye state-of-the-art lightweight model use kar rahe hain:

all-MiniLM-L6-v2

Features:
Embedding dimension: 384
Fast and lightweight
Excellent for:
Semantic search
Clustering
RAG
Recommendation systems

🧠 Real-Life Analogy

Ye model har review ko “meaning fingerprint” deta hai.

For example:

Review Text	Embedding
"Product key didn't work"	[0.123, -0.456, ...]
"Received a fake DVD"	[0.119, -0.449, ...]

Similar complaints ke vectors ek dusre ke paas hote hain.

Generate Embeddingsembeddings = model.encode(
   

In [6]:
embeddings = model.encode(
    documents,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings shape: (5000, 384)


Create ChromaDB Collection

In [7]:
# Path where ChromaDB will be stored
chroma_path = os.path.join(DATA_PATH, "chroma_db")

# Create persistent client
client = chromadb.PersistentClient(path=chroma_path)

# Create or load collection
collection = client.get_or_create_collection(
    name="negative_reviews"
)

print("ChromaDB collection created successfully!")
print("Database location:", chroma_path)

ChromaDB collection created successfully!
Database location: /content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/data/chroma_db


🏢 Real-Life Analogy

Imagine:

documents = books
embeddings = meaning fingerprints
collection = library shelf

Now all complaint reviews are ready to be stored in a searchable AI library.

Add document to ChromaDB

In [8]:
# Generate unique IDs for each document
ids = [f"doc_{i}" for i in range(len(documents))]

# Optional: If collection already contains data, clear it first
if collection.count() > 0:
    collection.delete(ids=collection.get()["ids"])
    print("Existing documents removed.")

# Add documents and embeddings to ChromaDB
collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    ids=ids
)

print("Documents added to ChromaDB successfully!")
print("Total documents in collection:", collection.count())

Documents added to ChromaDB successfully!
Total documents in collection: 5000


🧠 What This Cell Does

This cell:

Creates IDs like:
doc_0
doc_1
doc_2

Optionally removes old entries to avoid duplicate-ID errors
Inserts:
Review text
Embeddings
IDs

🏢 Real-Life Analogy

Imagine storing 5,000 complaint reports into a smart searchable library.

Each report now has:

Its text
Its semantic fingerprint
A unique identifier

So the system can instantly retrieve the most relevant complaints.

Perform your First Semantic Search

In [9]:
# Example query
query = "fake product and invalid key"

# Convert query to embedding
query_embedding = model.encode([query])

# Search in ChromaDB
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

# Display results
for i, doc in enumerate(results["documents"][0], 1):
    print("\n" + "=" * 80)
    print(f"RESULT {i}")
    print("=" * 80)
    print(doc[:1000])


RESULT 1
1st shipment received a book instead of the game.2nd shipment got a FAKE one. Game arrived with a wrong key inside on sealed box. I got in contact with codemasters and send them pictures of the DVD and the content. They said nothing they can do its a fake DVD.Returned it good bye.!

RESULT 2
I bought this and the key didn't work.  It was a gift, and the recipient wasn't able to solve the problem.  It might have been a good game, but I never found out because the key failed.

RESULT 3
decent attempt at a knock off, but its bootleg as hell. they should be selling these with crack so you can get something worth the money being spent.

RESULT 4
Products offered by many marketplace sellers are not original Nintendo Brand. Often they are unlicensed third party knock-off expansion packs. Ive personally had these damage a system and ruin games. Look for sellers offering the real thing by Nintendo. Ive warned you. You can save a few bucks and take your chances. Deviation from the actu

Create a Reusable Semantic Search Function

In [10]:
def semantic_search(query, n_results=5):
    """
    Search similar customer complaints using natural language.
    """

    # Convert query into embedding
    query_embedding = model.encode([query])

    # Search vector database
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )

    # Display results
    print(f"\n🔍 Query: {query}")
    print(f"📄 Top {n_results} matching reviews:\n")

    for i, doc in enumerate(results["documents"][0], 1):
        print("=" * 80)
        print(f"RESULT {i}")
        print("=" * 80)
        print(doc[:1000])
        print("\n")

    return results

In [11]:
semantic_search("product key not working")


🔍 Query: product key not working
📄 Top 5 matching reviews:

RESULT 1
I bought this and the key didn't work.  It was a gift, and the recipient wasn't able to solve the problem.  It might have been a good game, but I never found out because the key failed.


RESULT 2
Although I rather like the game and really enjoy the graphics and music of it, I now wish I hadn't purchased it at all. The game is rather buggy. It has a tendency to crash when you attempt to skip dialogue, and sometimes key items simply do not work. One of the online faqs I read explained in detail that sometimes an item simply won't work as it is supposed to, and in such a case, you basically have to start all over. Restarting your PSX/PS2 won't fix the problem. I had that happen 3 hours into the game with the library keys. What a waste of time. Good game; lousy quality control.


RESULT 3
I recieved the controller on time, how ever it is not fully functional, only the directional key works, and the start button works fr

{'ids': [['doc_4', 'doc_2338', 'doc_1250', 'doc_1', 'doc_3311']],
 'embeddings': None,
 'documents': [["I bought this and the key didn't work.  It was a gift, and the recipient wasn't able to solve the problem.  It might have been a good game, but I never found out because the key failed.",
   "Although I rather like the game and really enjoy the graphics and music of it, I now wish I hadn't purchased it at all. The game is rather buggy. It has a tendency to crash when you attempt to skip dialogue, and sometimes key items simply do not work. One of the online faqs I read explained in detail that sometimes an item simply won't work as it is supposed to, and in such a case, you basically have to start all over. Restarting your PSX/PS2 won't fix the problem. I had that happen 3 hours into the game with the library keys. What a waste of time. Good game; lousy quality control.",
   "I recieved the controller on time, how ever it is not fully functional, only the directional key works, and t

In [12]:
semantic_search("fake product")
semantic_search("installation problem")
semantic_search("game does not work")
semantic_search("broken item")
semantic_search("poor quality")
semantic_search("missing parts")
semantic_search("delivery issue")


🔍 Query: fake product
📄 Top 5 matching reviews:

RESULT 1
I did not get anything from this product personally and I haven't enjoyed this one at all.


RESULT 2
All that really mattered to me is that the product worked. You can tell that its a bootleg version. But for what it is. It works and it was cheap and i was happy with it.


RESULT 3
decent attempt at a knock off, but its bootleg as hell. they should be selling these with crack so you can get something worth the money being spent.


RESULT 4
not get clear about that game as nothing good, not work on code i try as nothing and is that fake or not kind code for this game? i don't understand


RESULT 5
1st shipment received a book instead of the game.2nd shipment got a FAKE one. Game arrived with a wrong key inside on sealed box. I got in contact with codemasters and send them pictures of the DVD and the content. They said nothing they can do its a fake DVD.Returned it good bye.!



🔍 Query: installation problem
📄 Top 5 matching rev

{'ids': [['doc_1159', 'doc_882', 'doc_2738', 'doc_1713', 'doc_2415']],
 'embeddings': None,
 'documents': [["Arrivedin timely manner, wrapped and packaged very well. The game was complete, came with everything. The game works just fine, except I'm unable to load a saved game.",
   "it arrived in great shape.  We have played it already and the only problem has been putting it in the machine it doesn't want to start right away.",
   "I've tried many times to play this game,and didn't get to even play it the first few times as it would just say that it's saving and would stay that way for an hour or more.Then I kept trying,and finally made it past certain points,but then the game screwed up again.So,I gave up on it.I am hoping that the seller wasn't being a snake and knew this game has a memory card glitch and still sold it.I wrote a review for this game a few days ago and I guess amazon wouldn't let me post for some reason.I was just trying to warn people of this game problem and possibl

🧠 Why Cell 8 Is So Important

Without Cell 8, every search requires multiple lines of code.

With Cell 8, you can simply write:

semantic_search("fake product")

And instantly retrieve the most relevant complaints.

This function becomes the core retrieval engine for:

RAG
Streamlit dashboard
FastAPI backend
Business Q&A system

In [13]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
print("API Key loaded successfully!" if api_key else "API Key not found.")

API Key loaded successfully!


🏆 Step 4: Build the RAG System
🎯 User Questions the System Will Answer
“What are the top customer complaints?”
“Why are customers unhappy?”
“What should the company improve first?”
“Summarize the major issues.”

How RAG Works
User Question
      ↓
Semantic Search (ChromaDB)
      ↓
Relevant Customer Reviews Retrieved
      ↓
OpenAI LLM
      ↓
Business Summary and Recommendations

In [14]:
# ==========================================
# Create 04_rag_with_openai.ipynb in Google Drive
# ==========================================

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import required libraries
import os
import json

# Define notebooks folder path
NOTEBOOKS_PATH = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/notebooks"

# Create notebooks folder if it doesn't exist
os.makedirs(NOTEBOOKS_PATH, exist_ok=True)

# Notebook file name
notebook_name = "04_rag_with_openai.ipynb"

# Full path to notebook
notebook_path = os.path.join(NOTEBOOKS_PATH, notebook_name)

# Minimal valid Jupyter notebook structure
notebook_content = {
    "cells": [],
    "metadata": {
        "colab": {
            "name": notebook_name,
            "provenance": []
        },
        "kernelspec": {
            "display_name": "Python 3",
            "name": "python3"
        },
        "language_info": {
            "name": "python"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 0
}

# Create the notebook file
with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(notebook_content, f, indent=2)

# Success message
print("Notebook created successfully! 🚀")
print("Location:", notebook_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook created successfully! 🚀
Location: /content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/notebooks/04_rag_with_openai.ipynb
